# 💻 Direct SQL Operations with Calliope

This notebook demonstrates how to use Calliope's SQL generation and execution capabilities for developers who want more control over their queries.

## Topics Covered
- Generate SQL without execution
- Run SQL directly
- Edit and refine generated SQL
- Combine natural language with direct SQL
- Debug and optimize queries

## 🗄️ Setup: Create Sample Database

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

conn = sqlite3.connect('hr_database.db')

# Employees table
employees = pd.DataFrame({
    'employee_id': range(1, 51),
    'name': [f'Employee_{i}' for i in range(1, 51)],
    'department': np.random.choice(['Engineering', 'Sales', 'Marketing', 'HR', 'Finance'], 50),
    'salary': np.random.randint(40000, 150000, 50),
    'hire_date': [(datetime(2020, 1, 1) + timedelta(days=np.random.randint(0, 1460))).strftime('%Y-%m-%d') for _ in range(50)],
    'manager_id': [np.random.choice([None, np.random.randint(1, 51)]) for _ in range(50)]
})

# Projects table
projects = pd.DataFrame({
    'project_id': range(1, 21),
    'project_name': [f'Project_{i}' for i in range(1, 21)],
    'budget': np.random.randint(50000, 500000, 20),
    'start_date': [(datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 365))).strftime('%Y-%m-%d') for _ in range(20)],
    'status': np.random.choice(['active', 'completed', 'planning'], 20)
})

# Project assignments
assignments = pd.DataFrame({
    'assignment_id': range(1, 101),
    'employee_id': np.random.randint(1, 51, 100),
    'project_id': np.random.randint(1, 21, 100),
    'hours_allocated': np.random.randint(10, 160, 100)
})

employees.to_sql('employees', conn, if_exists='replace', index=False)
projects.to_sql('projects', conn, if_exists='replace', index=False)
assignments.to_sql('assignments', conn, if_exists='replace', index=False)

conn.close()
print("✅ HR database created!")

In [ ]:
%%calliope add-database
{
    "datasource_id": "hr_db",
    "name": "HR Database",
    "dialect": "sqlite",
    "connection_details": {
        "database": "hr_database.db"
    }
}

## 🔧 Generate SQL (Without Execution)

Use `generate-sql` to create SQL from natural language without running it. Perfect for:
- Learning SQL
- Reviewing queries before execution
- Generating templates for modification

In [ ]:
%%calliope generate-sql hr_db
Show all employees in the Engineering department with their salaries

In [ ]:
%%calliope generate-sql hr_db --sql-model anthropic
Calculate the average salary by department and rank departments by average salary

In [ ]:
%%calliope generate-sql hr_db
Find employees who are working on more than 3 projects

## ⚡ Run SQL Directly

Use `run-sql` to execute your own SQL queries with Calliope's visualization capabilities:

In [ ]:
%%calliope run-sql hr_db
SELECT department, COUNT(*) as employee_count, AVG(salary) as avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC

In [ ]:
%%calliope run-sql hr_db
SELECT 
    e.name,
    e.department,
    COUNT(a.project_id) as project_count,
    SUM(a.hours_allocated) as total_hours
FROM employees e
JOIN assignments a ON e.employee_id = a.employee_id
GROUP BY e.employee_id, e.name, e.department
HAVING project_count > 2
ORDER BY total_hours DESC

## 🎯 Complex SQL Patterns

Generate complex SQL patterns using natural language:

In [ ]:
%%calliope generate-sql hr_db --sql-model anthropic
Using a CTE, find the top 3 highest paid employees in each department

In [ ]:
%%calliope generate-sql hr_db --sql-model anthropic
Use window functions to calculate the salary percentile for each employee within their department

In [ ]:
%%calliope generate-sql hr_db --sql-model anthropic
Show the organizational hierarchy using the manager_id relationship

## 🐛 Debugging and Optimization

Generate SQL with explanations to understand query logic:

In [ ]:
%%calliope generate-sql hr_db --sql-model anthropic --to-ai --model claude3
Explain how to find employees who have been with the company for more than 2 years and are working on active projects

## 🔄 Iterative Query Development

Workflow: Generate → Review → Modify → Execute

In [ ]:
# Step 1: Generate initial query
%%calliope generate-sql hr_db
Show project budgets and number of assigned employees

In [ ]:
# Step 2: Run the generated query (modify as needed)
%%calliope run-sql hr_db
SELECT 
    p.project_name,
    p.budget,
    p.status,
    COUNT(DISTINCT a.employee_id) as employee_count,
    SUM(a.hours_allocated) as total_hours
FROM projects p
LEFT JOIN assignments a ON p.project_id = a.project_id
GROUP BY p.project_id, p.project_name, p.budget, p.status
ORDER BY p.budget DESC

## 📊 Generate Query Summaries

Get AI-powered insights about your query results:

In [ ]:
%%calliope ask-sql hr_db --to-ai --model claude3
Analyze salary distribution across departments and identify any potential pay equity issues

In [ ]:
%%calliope generate-summary hr_db
Summarize the key insights from our employee and project data

## 🔗 Combining Natural Language and SQL

Use natural language for complex logic, then refine with direct SQL:

In [ ]:
# Generate the base query
%%calliope generate-sql hr_db --sql-model anthropic
Find employees with above-average salaries in their department who are working on at least one active project

In [ ]:
# Execute refined version
%%calliope run-sql hr_db
WITH dept_avg AS (
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
),
active_employees AS (
    SELECT DISTINCT e.employee_id
    FROM employees e
    JOIN assignments a ON e.employee_id = a.employee_id
    JOIN projects p ON a.project_id = p.project_id
    WHERE p.status = 'active'
)
SELECT 
    e.name,
    e.department,
    e.salary,
    da.avg_salary as dept_avg,
    ROUND((e.salary - da.avg_salary) / da.avg_salary * 100, 2) as pct_above_avg
FROM employees e
JOIN dept_avg da ON e.department = da.department
JOIN active_employees ae ON e.employee_id = ae.employee_id
WHERE e.salary > da.avg_salary
ORDER BY pct_above_avg DESC

## 💡 Pro Tips

### When to Use `generate-sql`
- Learning SQL syntax
- Creating query templates
- Understanding complex join logic
- Before executing on production data

### When to Use `run-sql`
- You already have SQL code
- Testing specific queries
- Debugging generated queries
- Performance-critical queries

### When to Use `ask-sql`
- Quick exploratory analysis
- Don't need to see SQL
- Want automatic visualizations
- Ad-hoc questions

## 🎯 Key Takeaways

- `generate-sql` creates SQL without executing it
- `run-sql` executes your custom SQL with visualizations
- Use `--sql-model anthropic` for complex query generation
- Combine natural language and direct SQL for best results
- Iterate: generate → review → modify → execute

## 📚 Next Steps

- **Calliope-Visualizations.ipynb** - Custom charts and dashboards
- **Calliope-MultiDatabase.ipynb** - Query multiple databases